In [1]:
import pandas as pd
import numpy as np

training_data = pd.read_parquet(
    "../../data/training_2025_player_games.parquet"
)

training_data.shape

(5610, 14)

In [2]:
features = [
    "target_share",
    "red_zone_share",
    "goal_line_share",
    "carries",
    "targets",
    "opportunities",
    "total_air_yards",
    "avg_air_yards",
    "avg_yardline_100"
]

target = "actual_fp_game"

In [3]:
X = training_data[features]
y = training_data[target]

X.head()

,target_share,red_zone_share,goal_line_share,carries,targets,opportunities,total_air_yards,avg_air_yards,avg_yardline_100
0,0.137931,0.333333,0.666667,12.0,4,16.0,-15.0,-3.750000,54.187500
1,0.034483,0.000000,0.000000,0.0,1,1.0,2.0,2.000000,37.000000
2,0.000000,0.111111,0.000000,7.0,0,7.0,0.0,0.000000,46.428571
3,0.034483,0.000000,0.000000,0.0,1,1.0,-2.0,-2.000000,30.000000
4,0.310345,0.111111,0.000000,0.0,9,9.0,35.0,3.888889,55.444444


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape

((4488, 9), (1122, 9))

In [5]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [6]:
y_pred = model.predict(X_test)

y_pred[:10]

array([ 1.7865    ,  3.214     ,  9.99452619,  5.0185    ,  1.144     ,
        6.6865    ,  5.1185    , 17.621     ,  2.7675    , 10.39723232])

In [7]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

mae, r2

(3.1018628274175564, 0.6221508267113284)

In [8]:
results = training_data.loc[X_test.index].copy()

results["xFP"] = y_pred
results["regression_delta"] = results["actual_fp_game"] - results["xFP"]

results[
    ["player_name", "actual_fp_game", "xFP", "regression_delta"]
].head(15)

,player_name,actual_fp_game,xFP,regression_delta
1020,L.Musgrave,1.6,1.786500,-0.186500
4038,C.Austin,4.1,3.214000,0.886000
3316,Mi.Wilson,13.5,9.994526,3.505474
2868,K.Pitts,3.4,5.018500,-1.618500
3447,M.Davis,2.4,1.144000,1.256000
5586,H.Henry,3.2,6.686500,-3.486500
4649,D.Goedert,12.2,5.118500,7.081500
1438,R.White,23.1,17.621000,5.479000
1235,T.Badie,2.6,2.767500,-0.167500
3262,M.Wilson,13.5,10.397232,3.102768


In [9]:
buy_low = results.sort_values(
    "regression_delta",
    ascending=True
)[
    ["player_name", "actual_fp_game", "xFP", "regression_delta"]
].head(15)

buy_low

,player_name,actual_fp_game,xFP,regression_delta
4423,D.London,5.7,21.026500,-15.326500
5295,K.Herbert,5.6,20.637143,-15.037143
798,M.Nabers,3.3,17.672000,-14.372000
3613,M.Tinsley,4.2,16.304800,-12.104800
3601,Z.Flowers,2.6,13.719000,-11.119000
181,J.Taylor,12.8,23.911000,-11.111000
2584,M.Harrison,12.3,23.384000,-11.084000
4483,C.Watson,3.7,14.510000,-10.810000
2948,A.Brown,11.9,22.622500,-10.722500
3753,J.Addison,8.6,19.017000,-10.417000


In [10]:
sell_high = results.sort_values(
    "regression_delta",
    ascending=False
)[
    ["player_name", "actual_fp_game", "xFP", "regression_delta"]
].head(15)

sell_high

,player_name,actual_fp_game,xFP,regression_delta
4588,A.Jeanty,37.8,13.437000,24.363000
2607,J.Taylor,49.6,27.415000,22.185000
3588,A.Brown,35.2,15.510500,19.689500
4568,K.Walker,25.4,7.458500,17.941500
2760,T.Henderson,28.0,10.414000,17.586000
915,J.Hill,28.7,11.584500,17.115500
1485,J.Croskey-Merritt,29.0,11.915500,17.084500
418,J.Taylor,29.5,14.336500,15.163500
4454,C.Brown,32.9,18.157000,14.743000
3488,J.Smith-Njigba,37.1,22.456500,14.643500


In [11]:
feature_importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance

,feature,importance
5,opportunities,0.477795
0,target_share,0.127508
4,targets,0.100465
8,avg_yardline_100,0.091523
1,red_zone_share,0.056466
6,total_air_yards,0.050476
7,avg_air_yards,0.050350
2,goal_line_share,0.027668
3,carries,0.017748


In [12]:
import joblib

joblib.dump(
    model,
    "../../ml/models/xfp_random_forest_v1.joblib"
)

['../../ml/models/xfp_random_forest_v1.joblib']

In [13]:
results.to_parquet(
    "../../data/xfp_test_predictions_2025.parquet",
    index=False
)